# Brillouin cascade: temperature-driven stochastic sweep

Edit `TEMPERATURES_K` in `scripts/nth_sweep.py`. For each temperature, the script evaluates

$$n_{\rm th}(T)=\frac{1}{\exp[\hbar\Omega_b/(k_B T)]-1},\qquad \Omega_b=2\pi\times6.02\,\mathrm{GHz},$$

and passes the resulting occupation to the SDE solver. At $T=0$, $n_{\rm th}=0$ exactly.

The physical defaults are $\gamma_j=2\pi\times83\,\mathrm{MHz}$ for every photon mode, $\Gamma=2\pi\times13.1\,\mathrm{MHz}$, and $g=11.1\,\mathrm{kHz}$.

In [ ]:
from pathlib import Path
import importlib
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from brillouin import plots as bp
importlib.reload(bp)

BACKEND = 'cuda'  # change to 'cuda' after running scripts/nth_sweep_cuda.py
SWEEP_JSON = ROOT / 'data' / ('nth_sweep_cuda_x2gap.json' if BACKEND == 'cuda' else 'nth_sweep.json')
S = bp.load_nth_sweep(SWEEP_JSON)

## 1. Generation curves

One separate figure per temperature; every figure contains the stochastic mean generation curves of all photon modes.

In [ ]:
generation_figures = bp.plot_temperature_generation(S)
for fig in generation_figures:
    fig.show()

## 2. Linewidths calculated from $g^{(1)}$

One separate figure per temperature; every figure contains `fwhm_g1` for all photon modes.

In [ ]:
linewidth_figures = bp.plot_temperature_linewidth_g1(S)
for fig in linewidth_figures:
    fig.show()

## 3. Stationary photon $g^{(2)}(0)$

One separate figure per temperature; every figure contains $g_j^{(2)}(0)$ for all photon modes. `amp_floor=0.0` keeps every calculated pump point.

In [ ]:
g2_figures = bp.plot_temperature_g2(S, amp_floor=0.0)
for fig in g2_figures:
    fig.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

a2_idx = 1  # a2: в Python это фотонная мода с индексом 1

temperatures = []
g2_a2_at_max_pump = []
max_pumps = []

for entry in S["entries"]:
    E = np.asarray(entry["E"], dtype=float)
    g2_a2 = np.asarray([
        np.nan if value is None else float(value)
        for value in entry["g2_0"][a2_idx]
    ])

    # Индекс максимального значения накачки
    max_pump_idx = int(np.nanargmax(E))

    temperatures.append(float(entry["T_K"]))
    max_pumps.append(E[max_pump_idx])
    g2_a2_at_max_pump.append(g2_a2[max_pump_idx])

temperatures = np.asarray(temperatures)
g2_a2_at_max_pump = np.asarray(g2_a2_at_max_pump)
max_pumps = np.asarray(max_pumps)

# Сортировка по температуре
order = np.argsort(temperatures)
temperatures = temperatures[order]
g2_a2_at_max_pump = g2_a2_at_max_pump[order]
max_pumps = max_pumps[order]

fig, ax = plt.subplots(figsize=(7, 4.5))

ax.plot(
    temperatures,
    g2_a2_at_max_pump,
    marker="o",
    linewidth=2,
    markersize=6,
    label=r"$g_{a_2}^{(2)}(0)$",
)

# Уровень когерентной статистики
ax.axhline(
    1.0,
    color="black",
    linestyle="--",
    linewidth=1,
    alpha=0.7,
    label=r"$g^{(2)}(0)=1$",
)

ax.set_xlabel(r"Temperature $T$ (K)")
ax.set_ylabel(r"$g_{a_2}^{(2)}(0; E_{\max})$")
ax.set_title(r"Second-order coherence of $a_2$ at maximum pump")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

plt.show()

for T, E_max, g2 in zip(
    temperatures,
    max_pumps,
    g2_a2_at_max_pump,
):
    print(f"T = {T:8.3f} K | E_max = {E_max:.6e} | g2_a2 = {g2:.6g}")